# NB07 — From crown segmentation to ecological forest metrics

**Branch:** `Musanga / Crown` · **Phase 1:** Baego ROI 1 and ROI 4 · **Runtime:** CPU.

**Scientific question:** Does good individual-crown segmentation accuracy translate into reliable forest-structure estimates?

The experiment uses **existing historical SAM1 ViT-H predictions only** (author-style and NMS), Kristof's extensive reference inventory, and Nadine's sparse reference inventory. No training or SAM inference. No input files are modified.

**Interpretation rules:** Nadine's sparse crowns serve crown-level geometric validation, **not stand-level density/cover reference**. Even Kristof's stand inventory requires a completeness audit before definitive bias claims. Crown centroid selection supports tree density; clipped footprint union supports canopy cover. **Summed CPA is not the same as union canopy cover**, and the ROI edge conventions differ for these metrics.

**Initial outputs:** a transparent per-ROI metric table, exploratory relative differences against Kristof annotations, and three figures.

In [ ]:
# 01 — Setup and read-only archive
from pathlib import Path
from google.colab import drive
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
DATA = ROOT / "data/Geo_Ref"
HISTORY = ROOT / "KVO_work/KVO_06_UNet_guided_SAM3/experiments_v53_luki_sam_only_replication"
RGB = DATA / "Baego_georef.tif"
CROWN_K = DATA / "Annotations/Crown_Kristof.shp"
CROWN_N = DATA / "Annotations/Crown_Nadine.shp"
ROI_PATH = DATA / "Annotations/ROI_Crowns_Kvo.shp"
MIN_REF_AREA_M2 = 5.0
ROI_IDS = (1, 4)
METHODS = ("author_style", "nms_060")

for p in [RGB, CROWN_K, CROWN_N, ROI_PATH, HISTORY]:
    if not p.exists(): raise FileNotFoundError(p)
with rasterio.open(RGB) as ds:
    CRS = ds.crs
    BOUNDS = ds.bounds
assert CRS is not None and CRS.is_projected, "Need metre-based projected raster CRS"
print("Working CRS:", CRS)
print("Archive:", HISTORY)

In [ ]:
# 02 — Load references, ROI polygons and existing SAM predictions
from shapely import make_valid

def polygonal(g):
    if g is None or g.is_empty: return Polygon()
    if not g.is_valid: g = make_valid(g)
    if isinstance(g, (Polygon, MultiPolygon)): return g
    parts = [x for x in getattr(g,"geoms",[]) if isinstance(x,(Polygon,MultiPolygon)) and not x.is_empty]
    return unary_union(parts) if parts else Polygon()

def read_projected(path, assume_missing_if_overlap=True):
    g = gpd.read_file(path)
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy()
    if g.crs is None:
        bx=g.total_bounds; rb=tuple(BOUNDS)
        overlaps=bx[2]>=rb[0] and bx[0]<=rb[2] and bx[3]>=rb[1] and bx[1]<=rb[3]
        if not (assume_missing_if_overlap and overlaps):
            raise ValueError(f"Unknown CRS and unmatched coordinates: {path}")
        print("WARNING: missing CRS; assuming raster CRS after XY bounds check:", path.name)
        g=g.set_crs(CRS)
    else:
        g=g.to_crs(CRS)
    g=g.reset_index(drop=True)
    g["geometry"]=gpd.GeoSeries([polygonal(x) for x in g.geometry],index=g.index,crs=CRS)
    return g.loc[~g.geometry.is_empty].reset_index(drop=True)

rois = read_projected(ROI_PATH)
kristof = read_projected(CROWN_K)
nadine = read_projected(CROWN_N)
kristof = kristof.loc[kristof.geometry.area>=MIN_REF_AREA_M2].reset_index(drop=True)
nadine = nadine.loc[nadine.geometry.area>=MIN_REF_AREA_M2].reset_index(drop=True)
prediction={}
for roi in ROI_IDS:
    if roi>=len(rois):raise ValueError(f"ROI index {roi} outside ROI file")
    for method in METHODS:
        path=HISTORY / f"roi{roi}_{method}.gpkg"
        if not path.exists():raise FileNotFoundError(path)
        prediction[(roi,method)]=read_projected(path)
print("Reference crowns (all Baego): Kristof",len(kristof),"Nadine",len(nadine))
for roi in ROI_IDS:
    print("ROI",roi,"area ha",round(rois.geometry.iloc[roi].area/1e4,3),
          "author",len(prediction[(roi,"author_style")]),
          "NMS",len(prediction[(roi,"nms_060")]))

In [ ]:
# 03 — Ecological metrics: centroid counts and spatially clipped canopy footprint
# Tree density / mean crown area: crowns with centroid inside ROI, whole crown area.
# Summed clipped CPA / union cover: all crowns that intersect ROI, clipped to ROI.
# These intentionally have different ROI-edge conventions; report them separately.
def metrics_for_roi(crowns, region):
    region_area_m2=region.area
    if region_area_m2<=0: raise ValueError("Zero-area ROI")
    centroid_mask=crowns.geometry.centroid.within(region)
    center_crowns=crowns.loc[centroid_mask]
    n=len(center_crowns)
    sizes=center_crowns.geometry.area.to_numpy()
    candidates=crowns.loc[crowns.geometry.intersects(region)]
    clipped=[polygonal(g.intersection(region)) for g in candidates.geometry]
    clipped=[g for g in clipped if not g.is_empty and g.area>0]
    sum_clipped=float(sum(g.area for g in clipped))
    canopy_union=float(unary_union(clipped).area) if clipped else 0.
    return {
        "roi_area_ha":region_area_m2/10000,
        "n_centroid":n,
        "density_ha":n/(region_area_m2/10000),
        "mean_crown_m2":float(np.mean(sizes)) if n else np.nan,
        "median_crown_m2":float(np.median(sizes)) if n else np.nan,
        "summed_whole_cpa_m2":float(sizes.sum()),
        "sum_clipped_cpa_m2":sum_clipped,
        "sum_clipped_cpa_ha_per_ha":sum_clipped/region_area_m2,
        "union_canopy_cover_pct":100*canopy_union/region_area_m2,
        "overlap_excess_pct_roi":100*(sum_clipped-canopy_union)/region_area_m2
    }

rows=[]
for roi in ROI_IDS:
    region=polygonal(rois.geometry.iloc[roi])
    for name,geom in [
        ("Kristof_reference",kristof),
        ("Nadine_sparse_NOT_STAND_REFERENCE",nadine),
        ("SAM1_author_style",prediction[(roi,"author_style")]),
        ("SAM1_nms_060",prediction[(roi,"nms_060")])
    ]:
        rows.append({"ROI":roi,"dataset":name,**metrics_for_roi(geom,region)})
METRIC_TABLE=pd.DataFrame(rows)
print("=== Descriptive ecological metrics — per Baego ROI ===")
display(METRIC_TABLE.round(3))
print("CAUTION: Nadine reference is sparse; do not compare stand metrics to it as a complete inventory.")

In [ ]:
# 04 — Exploratory SAM differences relative to Kristof annotations
# Differences are NOT validated stand-level biases until Kristof inventory completeness is checked.
cols=["density_ha","mean_crown_m2","median_crown_m2",
      "sum_clipped_cpa_ha_per_ha","union_canopy_cover_pct","overlap_excess_pct_roi"]
diff=[]
for roi in ROI_IDS:
    part=METRIC_TABLE.loc[METRIC_TABLE.ROI==roi].set_index("dataset")
    ref=part.loc["Kristof_reference"]
    for label in ("SAM1_author_style","SAM1_nms_060"):
        row={"ROI":roi,"method":label}
        for col in cols:
            value=part.loc[label,col]
            row[col+"_diff"]=value-ref[col]
            row[col+"_relative_pct"]=100*(value-ref[col])/ref[col] if ref[col] else np.nan
        diff.append(row)
DIFFERENCES=pd.DataFrame(diff)
show=["ROI","method"] + [c+"_relative_pct" for c in cols[:5]]
print("=== Relative difference from Kristof annotation estimates (%) ===")
display(DIFFERENCES[show].round(1))
print("These percentages are provisional comparisons, NOT ecological ground-truth bias.")

In [ ]:
# 05 — Figures: method comparison, crown-size distribution, aggregation sensitivity
from matplotlib.ticker import MaxNLocator

labels=["Kristof_reference","SAM1_author_style","SAM1_nms_060"]
names=["Reference","SAM1 original","SAM1 NMS"]
fig, axes=plt.subplots(1,3,figsize=(15,4.5),constrained_layout=True)
for ax,metric,title in zip(
        axes,
        ("density_ha","sum_clipped_cpa_ha_per_ha","union_canopy_cover_pct"),
        ("Crown density (ha⁻¹)","Summed clipped CPA (ha/ha)","Union canopy cover (%)")):
    for i,roi in enumerate(ROI_IDS):
        sub=METRIC_TABLE.loc[(METRIC_TABLE.ROI==roi)].set_index("dataset")
        values=[sub.loc[x,metric] for x in labels]
        x=np.arange(3)+i*3.7
        ax.bar(x,values,label=f"ROI {roi}" if ax==axes[0] else None)
        ax.set_xticks(np.r_[np.arange(3),np.arange(3)+3.7],
                      names*2,rotation=35,ha="right")
    ax.set_title(title);ax.grid(axis="y",alpha=.2)
axes[0].legend()
plt.show()

fig,axes=plt.subplots(1,2,figsize=(13,4),constrained_layout=True)
for ax,roi in zip(axes,ROI_IDS):
    region=rois.geometry.iloc[roi]
    for label,crowns in [
        ("Kristof",kristof),("SAM1 original",prediction[(roi,"author_style")]),
        ("SAM1 NMS",prediction[(roi,"nms_060")])
    ]:
        areas=crowns.loc[crowns.geometry.centroid.within(region)].geometry.area.to_numpy()
        areas=areas[areas>0]
        if len(areas):
            ax.hist(areas,bins=np.geomspace(2,max(2000.,areas.max()),25),
                    histtype="step",linewidth=2,label=label)
    ax.set_xscale("log");ax.set_title(f"ROI {roi}: crown-size distribution")
    ax.set_xlabel("Crown area (m²)");ax.set_ylabel("Number of crowns")
    ax.legend()
plt.show()

print("Note: scale sensitivity across nested spatial windows is a next step,")
print("after checking reference extent and ROI-area geometry. No inference used.")

## 06 — Decisions before extending the experiment

1. **Inventory completeness:** check Kristof annotation coverage within each ROI before interpreting predicted-minus-reference differences as bias.
2. **Reference protocol:** Nadine's sparse, high-quality crowns support IoU/shape validation but not unbiased density or canopy cover.
3. **Boundary effects:** clarify whether authors count crowns by centroid and whether polygons are clipped at plot boundaries; both choices influence density and CPA.
4. **Next analysis:** quantify individual-crown error types and check whether crown F1 predicts errors in density, summed CPA and union canopy cover across more independent, completely annotated ROIs.
5. **Then consider Luki:** use an equivalent frozen method and site-specific reference completeness audit.

**Frozen baseline:** historical SAM1 ViT-H Luki-style Block 53. This notebook is deliberately analysis-only.